In [1]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)

In [2]:
DATA_DIR = Path("data")
PREPARED_PATH = DATA_DIR / "prepared_year_after_preprocessing.csv"

MONTHS_RU = [
    "Январь", "Февраль", "Март", "Апрель", "Май", "Июнь",
    "Июль", "Август", "Сентябрь", "Октябрь", "Ноябрь", "Декабрь",
]

DATE_RAW = "DATE"
TIME_RAW = "TIME"
TARGET_MODE = "В сети"


@dataclass(frozen=True)
class StatsConfig:
    prepared_path: Path = PREPARED_PATH
    raw_duplicate_chunksize: int = 500_000
    raw_duplicate_nrows_per_month: int | None = None
    target_mode: str = TARGET_MODE


cfg = StatsConfig()
pd.DataFrame([asdict(cfg)])

,prepared_path,raw_duplicate_chunksize,raw_duplicate_nrows_per_month,target_mode
0,data\prepared_year_after_preprocessing.csv,500000,None,В сети


In [3]:
def month_file(month: str) -> Path:
    return DATA_DIR / month / "merged_with_mode.csv"


def build_timestamp(df: pd.DataFrame, date_col: str = DATE_RAW, time_col: str = TIME_RAW) -> pd.Series:
    date_part = (
        df[date_col]
        .astype(str)
        .str.replace("/", ".", regex=False)
        .str.strip()
    )
    time_part = (
        df[time_col]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .str.replace(" ", "", regex=False)
        .str.strip()
    )
    combined = date_part + " " + time_part
    parsed = pd.to_datetime(
        combined,
        format="%d.%m.%Y %H:%M:%S.%f",
        errors="coerce",
    )
    missing_mask = parsed.isna()
    if missing_mask.any():
        parsed.loc[missing_mask] = pd.to_datetime(
            combined.loc[missing_mask],
            format="%d.%m.%Y %H:%M:%S",
            errors="coerce",
        )
    return parsed


def iter_raw_timestamp_chunks(
    month: str,
    chunksize: int = cfg.raw_duplicate_chunksize,
    nrows: int | None = cfg.raw_duplicate_nrows_per_month,
):
    reader = pd.read_csv(
        month_file(month),
        sep=",",
        usecols=[DATE_RAW, TIME_RAW],
        chunksize=chunksize,
        dtype=str,
        encoding="utf-8",
        low_memory=False,
    )

    remaining = nrows
    for chunk in reader:
        if remaining is not None and remaining <= 0:
            break
        if remaining is not None and len(chunk) > remaining:
            yield chunk.iloc[:remaining].copy()
            break
        yield chunk
        if remaining is not None:
            remaining -= len(chunk)


def restore_timestamp_from_numeric(numeric_timestamp: pd.Series) -> pd.Series:
    valid = numeric_timestamp.dropna()
    if valid.empty:
        return pd.Series(pd.NaT, index=numeric_timestamp.index, dtype="datetime64[ns]")
    max_value = float(valid.max())
    if max_value < 1e11:
        unit = "s"
    elif max_value < 1e14:
        unit = "ms"
    elif max_value < 1e17:
        unit = "us"
    else:
        unit = "ns"
    return pd.to_datetime(numeric_timestamp, unit=unit, errors="coerce")


def load_prepared_year(path: Path = cfg.prepared_path) -> pd.DataFrame:
    prepared = pd.read_csv(path, encoding="utf-8", low_memory=False)
    timestamp_from_text = (
        pd.to_datetime(prepared["timestamp"], errors="coerce")
        if "timestamp" in prepared.columns
        else pd.Series(pd.NaT, index=prepared.index, dtype="datetime64[ns]")
    )
    if "timestamp_ms" in prepared.columns:
        prepared["timestamp_ms"] = pd.to_numeric(prepared["timestamp_ms"], errors="coerce").astype("Int64")
        timestamp_from_numeric = restore_timestamp_from_numeric(pd.to_numeric(prepared["timestamp_ms"], errors="coerce"))
        prepared["timestamp"] = (
            timestamp_from_text
            if timestamp_from_text.notna().sum() >= timestamp_from_numeric.notna().sum()
            else timestamp_from_numeric
        )
    else:
        prepared["timestamp"] = timestamp_from_text
    return prepared

In [4]:
def count_raw_duplicate_timestamps(
    months: list[str] = MONTHS_RU,
    chunksize: int = cfg.raw_duplicate_chunksize,
    nrows_per_month: int | None = cfg.raw_duplicate_nrows_per_month,
) -> pd.DataFrame:
    rows = []

    for month in months:
        total_rows = 0
        duplicate_rows = 0
        prev_ts = pd.NaT

        for chunk in iter_raw_timestamp_chunks(month, chunksize=chunksize, nrows=nrows_per_month):
            ts = build_timestamp(chunk)
            total_rows += len(ts)

            dup_mask = ts.eq(ts.shift())
            if not ts.empty and pd.notna(prev_ts):
                dup_mask.iloc[0] = ts.iloc[0] == prev_ts

            duplicate_rows += int(dup_mask.fillna(False).sum())
            if not ts.empty:
                prev_ts = ts.iloc[-1]

        rows.append(
            {
                "month": month,
                "raw_rows_scanned": total_rows,
                "raw_duplicate_timestamp_rows": duplicate_rows,
            }
        )

    return pd.DataFrame(rows)


def build_anomaly_method_stats(df: pd.DataFrame) -> pd.DataFrame:
    stage1 = df["stage1_anomaly"].fillna(False)
    stage2 = df["stage2_anomaly"].fillna(False)

    rows = [
        {"method": "Hampel only", "rows": int((stage1 & ~stage2).sum())},
        {"method": "Isolation Forest only", "rows": int((~stage1 & stage2).sum())},
        {"method": "Both", "rows": int((stage1 & stage2).sum())},
        {"method": "Any anomaly", "rows": int((stage1 | stage2).sum())},
    ]
    out = pd.DataFrame(rows)
    out["share"] = out["rows"] / len(df)
    return out


def build_main_stats(
    prepared: pd.DataFrame,
    raw_duplicate_stats: pd.DataFrame,
    target_mode: str = cfg.target_mode,
) -> pd.DataFrame:
    stage1 = prepared["stage1_anomaly"].fillna(False)
    stage2 = prepared["stage2_anomaly"].fillna(False)
    any_anomaly = prepared["stage_any_anomaly"].fillna(False)
    target_mask = prepared["mode"].eq(target_mode)

    rows = [
        {"metric": "prepared_rows_total", "value": int(len(prepared))},
        {"metric": "prepared_duplicate_timestamps", "value": int(prepared["timestamp"].duplicated().sum())},
        {"metric": "rows_not_target_mode", "value": int((~target_mask.fillna(False)).sum())},
        {"metric": "rows_with_original_gap", "value": int(prepared["is_gap"].fillna(False).sum())},
        {"metric": "rows_imputed_short_gap", "value": int(prepared["imputed_short_gap"].fillna(False).sum())},
        {"metric": "rows_with_long_gap", "value": int(prepared["has_long_gap"].fillna(False).sum())},
        {"metric": "remaining_missing_cells_after_processing", "value": int(prepared.isna().sum().sum())},
        {"metric": "rows_flagged_by_hampel", "value": int(stage1.sum())},
        {"metric": "rows_flagged_by_isolation_forest", "value": int(stage2.sum())},
        {"metric": "rows_with_any_anomaly", "value": int(any_anomaly.sum())},
        {"metric": "raw_duplicate_timestamp_rows", "value": int(raw_duplicate_stats["raw_duplicate_timestamp_rows"].sum())},
    ]
    return pd.DataFrame(rows)


def build_monthly_stats(
    prepared: pd.DataFrame,
    raw_duplicate_stats: pd.DataFrame,
    target_mode: str = cfg.target_mode,
) -> pd.DataFrame:
    work = prepared.copy()
    work["month"] = work["timestamp"].dt.month

    grouped = (
        work.groupby("month", dropna=False)
        .agg(
            prepared_rows=("timestamp", "size"),
            not_target_mode_rows=("mode", lambda s: int((~s.eq(target_mode).fillna(False)).sum())),
            original_gap_rows=("is_gap", lambda s: int(s.fillna(False).sum())),
            imputed_short_gap_rows=("imputed_short_gap", lambda s: int(s.fillna(False).sum())),
            long_gap_rows=("has_long_gap", lambda s: int(s.fillna(False).sum())),
            hampel_rows=("stage1_anomaly", lambda s: int(s.fillna(False).sum())),
            isolation_rows=("stage2_anomaly", lambda s: int(s.fillna(False).sum())),
            any_anomaly_rows=("stage_any_anomaly", lambda s: int(s.fillna(False).sum())),
        )
        .reset_index()
    )

    month_number_map = {i + 1: month for i, month in enumerate(MONTHS_RU)}
    grouped["month_name"] = grouped["month"].map(month_number_map)
    grouped = grouped.merge(
        raw_duplicate_stats,
        left_on="month_name",
        right_on="month",
        how="left",
        suffixes=("", "_raw"),
    )
    grouped.drop(columns=["month_raw"], inplace=True)
    return grouped

In [5]:
prepared = load_prepared_year(cfg.prepared_path)
raw_duplicate_stats = count_raw_duplicate_timestamps(
    months=MONTHS_RU,
    chunksize=cfg.raw_duplicate_chunksize,
    nrows_per_month=cfg.raw_duplicate_nrows_per_month,
)

summary_stats = build_main_stats(prepared, raw_duplicate_stats, target_mode=cfg.target_mode)
anomaly_method_stats = build_anomaly_method_stats(prepared)
monthly_stats = build_monthly_stats(prepared, raw_duplicate_stats, target_mode=cfg.target_mode)

display(summary_stats)
display(anomaly_method_stats)
display(raw_duplicate_stats)
display(monthly_stats)

,metric,value
0,prepared_rows_total,30240001
1,prepared_duplicate_timestamps,0
2,rows_not_target_mode,5774133
3,rows_with_original_gap,0
4,rows_imputed_short_gap,0
5,rows_with_long_gap,0
6,remaining_missing_cells_after_processing,5854574
7,rows_flagged_by_hampel,2037181
8,rows_flagged_by_isolation_forest,166464
9,rows_with_any_anomaly,2189005


,method,rows,share
0,Hampel only,2037181,0.067367
1,Isolation Forest only,151824,0.005021
2,Both,14640,0.000484
3,Any anomaly,2189005,0.072388


,month,raw_rows_scanned,raw_duplicate_timestamp_rows
0,Январь,1382408,6
1,Февраль,2419214,9
2,Март,2678416,8
3,Апрель,2592015,12
4,Май,2678416,13
5,Июнь,2592015,12
6,Июль,2678410,7
7,Август,2678416,11
8,Сентябрь,2592015,11
9,Октябрь,2678416,15


,month,prepared_rows,not_target_mode_rows,original_gap_rows,imputed_short_gap_rows,long_gap_rows,hampel_rows,isolation_rows,any_anomaly_rows,month_name,raw_rows_scanned,raw_duplicate_timestamp_rows
0,1,1382401,658986,0,0,0,91611,66763,150492,Январь,1382408,6
1,2,2419200,642700,0,0,0,165030,33853,196178,Февраль,2419214,9
2,3,2678400,43,0,0,0,178279,0,178279,Март,2678416,8
3,4,2592000,1079002,0,0,0,171453,20834,190992,Апрель,2592015,12
4,5,2678400,685774,0,0,0,196524,1507,197881,Май,2678416,13
5,6,2592000,1330392,0,0,0,157420,19614,176005,Июнь,2592015,12
6,7,2678400,70,0,0,0,251431,0,251431,Июль,2678410,7
7,8,2678400,42,0,0,0,244723,0,244723,Август,2678416,11
8,9,2592000,1376982,0,0,0,90898,23893,113212,Сентябрь,2592015,11
9,10,2678400,56,0,0,0,170400,0,170400,Октябрь,2678416,15
